# Zero shot

Zero-shot text-to-tracklet retrieval on TVPReid test for X-CLIP, LanguageBind, InternVideo2 CLIP-S, and InternVideo2-1B-s2.

Clones the shawaf repo and OpenGVLab/InternVideo. Enable GPU and Internet. The 1B-s2 checkpoint is gated: accept the license on Hugging Face, then attach a Kaggle secret named `hugging_face` with your token. The install cell loads that secret.

CLIP-S, X-CLIP, and LanguageBind use 8-frame clips. 1B-s2 uses 4-frame clips (its native length), fp16, and batch 1. Each model is scored with a uniform sample and with sliding-window protocols. Window rows report the pool (`mean`, `mean_s8`, `max`, `query_max`). The last cell prints one table of every metric plus the sampling or window settings.


In [ ]:
from pathlib import Path

MODELS = ["xclip", "languagebind", "internvideo2", "internvideo2_s2_1b"]
SUBSETS = ["prid", "ilids", "duke"]
BATCH = 4
TEXT_BATCH = 16
DEVICE = "cuda"
SPLIT = "test"
POOLS = ("mean", "mean_s8", "max", "query_max")

# 8-frame encoders. 1B-s2 is built for 4 frames, so its window length is 4.
CLIP8 = {"xclip", "languagebind", "internvideo2"}
NATIVE_FRAMES = {"internvideo2_s2_1b": 4}
MODEL_BATCH = {"internvideo2_s2_1b": 1, "internvideo2": 2, "languagebind": 2}

SINGLE_PROTOCOLS = [
    {"name": "uniform8", "num_frames": 8, "frame_sample": "uniform", "models": "clip8"},
    {"name": "uniform4", "num_frames": 4, "frame_sample": "uniform", "models": "s2"},
    {"name": "middle4", "num_frames": 4, "frame_sample": "middle", "models": "s2"},
]
WINDOW_PROTOCOLS = [
    {"name": "vt_1fps_n12", "sample_fps": 1.0, "max_frames": 12, "stride": 4},
    {"name": "vt_2fps_n32", "sample_fps": 2.0, "max_frames": 32, "stride": 4},
    {"name": "reid_8fps_n64", "sample_fps": 8.0, "max_frames": 64, "stride": 4},
]

RESULTS_DIR = Path("/kaggle/working/zero_shot_results") if Path("/kaggle/working").is_dir() else Path("results/zero_shot")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Zero shot", MODELS, SUBSETS, flush=True)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

SHAWAF_URL = "https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git"
INTERNVIDEO_URL = "https://github.com/OpenGVLab/InternVideo.git"
EXPECTED = "0.1.16"


def run(cmd, cwd=None):
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)


ON_KAGGLE = Path("/kaggle/working").is_dir()
if ON_KAGGLE:
    shawaf_dir = Path("/kaggle/working/shawaf-vlm")
    intern_dir = Path("/kaggle/working/InternVideo")
    if shawaf_dir.exists():
        run(["git", "fetch", "origin"], cwd=shawaf_dir)
        run(["git", "reset", "--hard", "origin/main"], cwd=shawaf_dir)
    else:
        run(["git", "clone", SHAWAF_URL, str(shawaf_dir)])
    if intern_dir.exists():
        run(["git", "fetch", "origin"], cwd=intern_dir)
        run(["git", "reset", "--hard", "origin/main"], cwd=intern_dir)
    else:
        run(["git", "clone", INTERNVIDEO_URL, str(intern_dir)])
    run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{shawaf_dir}[all]"])
    run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "einops", "timm", "av", "imageio", "librosa", "soundfile",
            "pandas", "pyyaml", "scipy", "wandb",
        ]
    )
    try:
        run([sys.executable, "-m", "pip", "install", "-q", "decord"])
    except subprocess.CalledProcessError:
        print("decord wheel failed; the fine-tune path stubs it and uses PyAV", flush=True)
else:
    here = Path.cwd().resolve()
    shawaf_dir = here
    for candidate in [here, *here.parents]:
        if (candidate / "shawaf_vlm").is_dir() and (candidate / "pyproject.toml").is_file():
            shawaf_dir = candidate
            break
    intern_dir = shawaf_dir / "InternVideo"
    if not intern_dir.is_dir():
        run(["git", "clone", INTERNVIDEO_URL, str(intern_dir)], cwd=shawaf_dir)

os.environ["INTERNVIDEO_ROOT"] = str(intern_dir)
if str(shawaf_dir) not in sys.path:
    sys.path.insert(0, str(shawaf_dir))

for name in list(sys.modules):
    if name == "shawaf_vlm" or name.startswith("shawaf_vlm."):
        del sys.modules[name]

import shawaf_vlm

print("shawaf_vlm", shawaf_vlm.__version__, shawaf_vlm.__file__, flush=True)
print("INTERNVIDEO_ROOT", os.environ["INTERNVIDEO_ROOT"], flush=True)
if shawaf_vlm.__version__ != EXPECTED:
    raise RuntimeError(
        f"Expected shawaf_vlm {EXPECTED}, found {shawaf_vlm.__version__}. "
        "Restart the session and run this cell again after origin/main updates."
    )

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not token and Path("/kaggle/working").is_dir():
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("hugging_face")
if not token:
    raise RuntimeError(
        "Hugging Face token missing. On Kaggle, add a secret named hugging_face "
        "and attach it to this notebook."
    )
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
from huggingface_hub import login

login(token=token, add_to_git_credential=False)
print("Hugging Face token loaded", flush=True)


In [ ]:
from shawaf_vlm.data.tvpreid import download_tvpreid, load_tvpreid

DATA_ROOT = download_tvpreid(configs=tuple(SUBSETS), split=SPLIT)
print("TVPReid", SPLIT, DATA_ROOT, flush=True)


In [ ]:
import gc
import json

from shawaf_vlm.eval_loop import evaluate_text_to_tracklet, evaluate_text_to_tracklet_windows
from shawaf_vlm.metrics import format_metrics
from shawaf_vlm.models import build_encoder
from shawaf_vlm.models.runtime import ensure_cuda_healthy

METRIC_KEYS = [
    "Rank-1", "Rank-5", "Rank-10", "Rank-20", "Rank-50",
    "mAP", "MdR", "MnR", "nDCG@10", "mINP",
    "num_valid_queries", "num_gallery", "num_clips",
    "decode_s", "video_s", "text_s", "score_s", "total_s",
    "video_ms_per_item", "peak_gpu_gb", "reserved_gpu_gb",
]


def _jsonable(metrics):
    out = {}
    for key in METRIC_KEYS:
        if key not in metrics:
            continue
        value = metrics[key]
        out[key] = float(value) if value is not None else None
    return out


def _record(model, subset, protocol, pool, sampling, metrics):
    row = {
        "model": model,
        "subset": subset,
        "protocol": protocol,
        "pool": pool,
        "sampling": sampling,
    }
    row.update(_jsonable(metrics))
    rows.append(row)
    safe = protocol.replace("/", "-")
    out = RESULTS_DIR / f"{model}_tvpreid_{subset}_{safe}_{pool}.json"
    out.write_text(json.dumps(row, indent=2), encoding="utf-8")
    print(format_metrics(metrics), flush=True)
    print("Wrote", out, flush=True)


ensure_cuda_healthy(DEVICE)
rows = []
frame_cache = DATA_ROOT / "frame_cache"

for name in MODELS:
    native = NATIVE_FRAMES.get(name, 8)
    batch = MODEL_BATCH.get(name, BATCH)
    encoder = build_encoder(name, device=DEVICE)
    family = "clip8" if name in CLIP8 else "s2"
    for subset in SUBSETS:
        splits = load_tvpreid(subset, split=SPLIT, root=DATA_ROOT)
        print(
            f"{name} {subset}: {len(splits.query)} queries, {len(splits.gallery)} gallery",
            flush=True,
        )
        for spec in SINGLE_PROTOCOLS:
            if spec["models"] != family:
                continue
            sampling = f"{spec['frame_sample']} {spec['num_frames']} frames"
            print(f"== {name} {subset} {spec['name']} ({sampling})", flush=True)
            metrics = evaluate_text_to_tracklet(
                encoder,
                splits,
                num_frames=spec["num_frames"],
                batch_size=batch,
                text_batch_size=TEXT_BATCH,
                junk_same_camera=False,
                frame_cache=frame_cache,
                frame_sample=spec["frame_sample"],
            )
            _record(name, subset, spec["name"], "none", sampling, metrics)
        for spec in WINDOW_PROTOCOLS:
            sampling = (
                f"sliding windows of {native}, stride {spec['stride']}, "
                f"{spec['sample_fps']:g} fps, cap {spec['max_frames']} frames"
            )
            print(f"== {name} {subset} {spec['name']} ({sampling})", flush=True)
            pooled = evaluate_text_to_tracklet_windows(
                encoder,
                splits,
                num_frames=native,
                stride=spec["stride"],
                sample_fps=spec["sample_fps"],
                max_frames=spec["max_frames"],
                pools=POOLS,
                batch_size=batch,
                text_batch_size=TEXT_BATCH,
                junk_same_camera=False,
                frame_cache=frame_cache,
            )
            for pool, metrics in pooled.items():
                print(f"-- pool {pool}", flush=True)
                _record(name, subset, spec["name"], pool, sampling, metrics)
    del encoder
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

print(f"Collected {len(rows)} rows", flush=True)


In [ ]:
import pandas as pd

TABLE_COLUMNS = [
    "model", "subset", "protocol", "pool", "sampling",
    "Rank-1", "Rank-5", "Rank-10", "Rank-20", "Rank-50",
    "mAP", "MdR", "MnR", "nDCG@10", "mINP",
    "num_valid_queries", "num_gallery", "num_clips",
    "decode_s", "video_s", "text_s", "score_s", "total_s",
    "video_ms_per_item", "peak_gpu_gb", "reserved_gpu_gb",
]
table = pd.DataFrame(rows)
for column in TABLE_COLUMNS:
    if column not in table.columns:
        table[column] = pd.NA
table = table[TABLE_COLUMNS].sort_values(
    ["model", "subset", "protocol", "pool"],
    kind="mergesort",
)
table_path = RESULTS_DIR / "zero_shot_table.csv"
table.to_csv(table_path, index=False)

pd.set_option("display.max_rows", 400)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 80)
float_cols = [
    "Rank-1", "Rank-5", "Rank-10", "Rank-20", "Rank-50",
    "mAP", "MdR", "MnR", "nDCG@10", "mINP",
    "decode_s", "video_s", "text_s", "score_s", "total_s",
    "video_ms_per_item", "peak_gpu_gb", "reserved_gpu_gb",
]
shown = table.copy()
for column in float_cols:
    shown[column] = pd.to_numeric(shown[column], errors="coerce").round(2)
print(shown.to_string(index=False))
print("Wrote", table_path, flush=True)

best = (
    table.sort_values("Rank-1", ascending=False, kind="mergesort")
    .groupby(["model", "subset"], as_index=False)
    .head(1)
)
print("\nBest Rank-1 per model and subset")
print(
    best[["model", "subset", "protocol", "pool", "sampling", "Rank-1", "mAP", "MdR"]]
    .to_string(index=False)
)
